# Server Backend YOLO + XAI (Google Colab GPU)
### Sistem Analisis Investigasi Kecelakaan - Program Tesis S2

Notebook ini menjalankan backend komputasi deteksi objek YOLOv8, Grad-CAM, LIME, SHAP, dan BLIP Image Captioning.

**Langkah Penggunaan:**
1. Pastikan Runtime menggunakan GPU: **Runtime -> Change runtime type -> T4 GPU**.
2. Jalankan semua sel: **Runtime -> Run all** (atau tekan Ctrl + F9).
3. Izinkan akses Google Drive saat diminta.
4. Tunggu hingga sel terakhir menampilkan `SERVER AKTIF & SIAP MENERIMA PERMINTAAN`.

In [ ]:
# 1. Pemasangan Dependensi & Library Machine Learning
!pip -q install ultralytics flask flask-cors pyngrok lime shap scikit-image transformers sentencepiece sacremoses
print('1/5 Dependensi berhasil dipasang.')

In [ ]:
# 2. Menghubungkan Google Drive & Memuat Model
import os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/Program Tesis Colab')
LOCAL_MODEL_PATH = Path('/content/best.pt')

if (DRIVE_PROJECT_DIR / 'best.pt').exists():
    MODEL_PATH = DRIVE_PROJECT_DIR / 'best.pt'
    print(f'2/5 Model ditemukan di Google Drive: {MODEL_PATH}')
elif LOCAL_MODEL_PATH.exists():
    MODEL_PATH = LOCAL_MODEL_PATH
    print(f'2/5 Model ditemukan di /content/best.pt')
else:
    raise FileNotFoundError(
        'File model best.pt tidak ditemukan di Drive (folder: Program Tesis Colab) maupun /content/.'
    )

In [ ]:
# 3. Menyiapkan Kode Backend Server (shap_server.py)
import sys, shutil
from pathlib import Path

DRIVE_SERVER_PATH = Path('/content/drive/MyDrive/Program Tesis Colab/shap_server.py')
if DRIVE_SERVER_PATH.exists():
    shutil.copy2(DRIVE_SERVER_PATH, '/content/shap_server.py')

if '/content' not in sys.path:
    sys.path.insert(0, '/content')

from shap_server import create_app
print('3/5 Modul backend shap_server.py siap.')

In [ ]:
# 4. Konfigurasi Authtoken & Domain Ngrok
import os
from pyngrok import ngrok

token = None
try:
    from google.colab import userdata
    token = userdata.get('NGROK_AUTHTOKEN')
except Exception:
    pass

if not token:
    token = os.environ.get('NGROK_AUTHTOKEN')

if not token:
    token = input('Masukkan NGROK_AUTHTOKEN Anda: ').strip()

assert token, 'NGROK_AUTHTOKEN wajib diisi!'
ngrok.set_auth_token(token)

domain = None
try:
    domain = userdata.get('NGROK_DOMAIN')
except Exception:
    pass

if domain:
    domain = domain.replace('https://', '').replace('http://', '').rstrip('/')
    print(f'4/5 Menggunakan Static Domain Ngrok: {domain}')
else:
    print('4/5 Menggunakan Dynamic Tunnel Ngrok.')

In [ ]:
# 5. Menjalankan Server YOLO & Membuka Akses Publik (Ngrok)
import threading, time, requests, torch
from shap_server import create_app

device_info = 'T4 GPU' if torch.cuda.is_available() else 'CPU'
print(f'Komputasi berjalan pada: {device_info}')

app = create_app(str(MODEL_PATH))
server_thread = threading.Thread(
    target=lambda: app.run(host='0.0.0.0', port=5000, use_reloader=False, threaded=True),
    daemon=True
)
server_thread.start()
time.sleep(3)

ngrok.kill()
try:
    if domain:
        public_url = ngrok.connect(addr=5000, bind_tls=True, domain=domain).public_url
    else:
        public_url = ngrok.connect(addr=5000, bind_tls=True).public_url
except Exception as e:
    print(f'Koneksi dengan domain khusus ({domain}) gagal ({e}), beralih ke dynamic tunnel...')
    public_url = ngrok.connect(addr=5000, bind_tls=True).public_url

health_res = requests.get(public_url + '/halo', headers={'ngrok-skip-browser-warning': '1'}, timeout=30).json()

print('=' * 75)
print('SERVER AKTIF & SIAP MENERIMA PERMINTAAN!')
print(f'URL Backend Publik : {public_url}')
print(f'Device Status      : {health_res.get("device")}')
print(f'Daftar Kelas       : {health_res.get("kelas")}')
print('=' * 75)
print('Catatan: Biarkan notebook ini tetap terbuka selama aplikasi digunakan.\n')

while True:
    time.sleep(60)